In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ===============================================================
# PHASE 0 - PROJECT SETUP & GOVERNANCE
# Applied Machine Learning Project | Medical Cost Prediction
# ===============================================================

# --- Install required packages (uncomment if needed) ---
# !pip install numpy pandas scikit-learn xgboost lightgbm catboost optuna shap matplotlib plotly imbalanced-learn

# --- Imports ---
import os
import json
import math
import random
import warnings
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd

# Plotting libraries
import matplotlib.pyplot as plt
import plotly.express as px

# Sklearn config
from sklearn import set_config
set_config(transform_output="pandas")

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)

In [ ]:
# ===============================================================
# 0.1 Reproducibility setup
# ===============================================================

SEED = 42

def set_seeds(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    return seed

_ = set_seeds(SEED)
print(f"Random seed set to {SEED}")

In [ ]:
# ===============================================================
# 0.2 Project directories
# ===============================================================

PROJECT_ROOT = Path.cwd()  # current working directory (where notebook is)
DATA_DIR     = PROJECT_ROOT / "data"
RAW_DIR      = DATA_DIR / "raw"
INT_DIR      = DATA_DIR / "interim"
PROC_DIR     = DATA_DIR / "processed"
MODEL_DIR    = PROJECT_ROOT / "models"
REPORT_DIR   = PROJECT_ROOT / "reports"
FIG_DIR      = REPORT_DIR / "figures"
TABLE_DIR    = REPORT_DIR / "tables"
SRC_DIR      = PROJECT_ROOT / "src"

for p in [DATA_DIR, RAW_DIR, INT_DIR, PROC_DIR, MODEL_DIR, REPORT_DIR, FIG_DIR, TABLE_DIR, SRC_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("✅ Project directories created successfully.")


In [ ]:
# ===============================================================
# 0.3 Load dataset
# ===============================================================

# IMPORTANT: Replace "Your_Folder_Path/" with the actual path to your file
# inside 'My Drive'. For example: 'Colab Notebooks/data/'
csv_path = "/content/drive/MyDrive/medical_insurance.csv"
# A common path structure is /content/drive/MyDrive/Your_Folder_Name/file_name.csv

assert Path(csv_path).exists(), f"CSV not found: {csv_path}"
df_raw = pd.read_csv(csv_path)
print(f"✅ Dataset loaded successfully: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns.")
df_raw.head(3)

In [ ]:
# ===============================================================
# 0.4 Schema snapshot
# ===============================================================

def quick_schema(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "n_missing": df.isna().sum().values,
        "pct_missing": (df.isna().mean().values * 100).round(2),
        "n_unique": [df[c].nunique(dropna=True) for c in df.columns],
        "example": [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns]
    })

schema = quick_schema(df_raw)
schema.to_csv(REPORT_DIR / "schema_snapshot.csv", index=False)
display(schema.head(20))

In [ ]:
# ===============================================================
# 0.5 Data card metadata
# ===============================================================

data_card = {
    "name": "Medical Insurance Cost & Risk Dataset",
    "records": int(df_raw.shape[0]),
    "features": int(df_raw.shape[1]),
    "source": "Synthetic dataset for Applied Machine Learning project",
    "intended_use": [
        "Regression: annual medical expenditure prediction",
        "Classification: cost-based risk tiering",
        "Unsupervised: clustering, PCA for persona/feature discovery"
    ],
    "sensitive_notes": "No PHI expected. Treat demographics responsibly; evaluate fairness.",
    "target_candidates": ["annual_cost", "total_medical_expenditure", "charges"],
    "last_updated": pd.Timestamp.today().strftime("%Y-%m-%d")
}

with open(REPORT_DIR / "data_card.json", "w") as f:
    json.dump(data_card, f, indent=2)

print("✅ Data card saved.")
print(json.dumps(data_card, indent=2))

In [ ]:
# ===============================================================
# 0.6 Utility: deterministic train/valid/test split indices
# ===============================================================

from sklearn.model_selection import train_test_split

def save_indices(y_like: pd.Series, test_size=0.2, valid_size=0.2, seed=SEED, stratify_bins: int | None = 10):
    """
    Returns and saves index splits: train_idx, valid_idx, test_idx
    - For regression: stratify by binned target if provided.
    - For classification: pass y_like as class labels (stratify directly).
    """
    idx = np.arange(len(y_like))
    # Regression stratification
    if pd.api.types.is_numeric_dtype(y_like) and stratify_bins:
        y_for_strat = pd.qcut(y_like.rank(method="first"), q=stratify_bins, labels=False, duplicates="drop")
    else:
        y_for_strat = y_like

    idx_tr, idx_te = train_test_split(idx, test_size=test_size, random_state=seed, stratify=y_for_strat)

    y_tr = y_like.iloc[idx_tr]
    if pd.api.types.is_numeric_dtype(y_tr) and stratify_bins:
        y_for_strat_tr = pd.qcut(y_tr.rank(method="first"), q=stratify_bins, labels=False, duplicates="drop")
    else:
        y_for_strat_tr = y_tr

    idx_tr, idx_va = train_test_split(idx_tr, test_size=valid_size, random_state=seed, stratify=y_for_strat_tr)

    np.save(PROC_DIR / "idx_train.npy", idx_tr)
    np.save(PROC_DIR / "idx_valid.npy", idx_va)
    np.save(PROC_DIR / "idx_test.npy", idx_te)

    print("✅ Indices saved to processed/ folder.")
    return idx_tr, idx_va, idx_te

print("Split utility ready for later use.")

In [ ]:
# ===============================================================
# 0.7 Plotting helpers
# ===============================================================

def dist_plot(series: pd.Series, title: str = "", log=False, bins=50):
    s = series.dropna()
    if log:
        s = np.log1p(s.clip(lower=0))
        title = f"{title} (log1p)"
    plt.figure(figsize=(6,4))
    plt.hist(s, bins=bins)
    plt.title(title)
    plt.xlabel(series.name)
    plt.ylabel("count")
    plt.show()

def bar_topk_counts(series: pd.Series, k=20, title="Top categories"):
    vc = series.value_counts(dropna=False).head(k)
    plt.figure(figsize=(7,4))
    vc.plot(kind="bar")
    plt.title(title)
    plt.ylabel("count")
    plt.show()

print("✅ Plotting helpers ready.")
print("Phase 0 complete — environment initialized successfully.")

In [ ]:
# ===============================================================
# PHASE 1 - EXPLORATORY DATA ANALYSIS (EDA)
# ===============================================================

# Make a working copy so original raw data stays untouched
df = df_raw.copy()

print("Dataset shape:", df.shape)
print("\n--- Basic Info ---")
df.info()

print("\n--- First 5 rows ---")
display(df.head())


In [ ]:
# ===============================================================
# Missing Values & Summary Statistics
# ===============================================================

# Missing value summary
missing_summary = (
    df.isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "column", 0: "n_missing"})
)
missing_summary["pct_missing"] = (missing_summary["n_missing"] / len(df) * 100).round(2)
missing_summary = missing_summary[missing_summary["n_missing"] > 0]

if len(missing_summary) == 0:
    print("✅ No missing values found in the dataset.")
else:
    print("⚠️ Missing values detected:")
    display(missing_summary.sort_values("pct_missing", ascending=False))

# Numeric summary
print("\n--- Numeric Feature Summary ---")
display(df.describe().T)

# Categorical summary (for reference)
print("\n--- Categorical Feature Overview ---")
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
for c in cat_cols:
    n_unique = df[c].nunique(dropna=True)
    print(f"{c:25s} | unique values: {n_unique}")


In [ ]:
# ===============================================================
# Numerical Feature Distributions & Outlier Detection
# ===============================================================

# Identify numeric columns
num_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
print(f"Numeric columns ({len(num_cols)}): {num_cols}\n")

# Plot histograms for numeric variables
for col in num_cols:
    print(f"🔹 Distribution for: {col}")
    dist_plot(df[col], title=f"{col} distribution", log=False, bins=40)

# Optional: also inspect log-scale if values vary widely (e.g., cost variables)
possible_targets = ["annual_cost", "charges", "expenses", "medical_cost", "expenditure"]
for tgt in possible_targets:
    if tgt in df.columns:
        print(f"\n🔹 Log-scale distribution for: {tgt}")
        dist_plot(df[tgt], title=f"{tgt} (log1p scale)", log=True, bins=40)

# Quick numeric outlier check using IQR rule
outlier_summary = {}
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outlier_pct = ((df[col] < lower) | (df[col] > upper)).mean() * 100
    outlier_summary[col] = round(outlier_pct, 2)

outlier_df = pd.DataFrame.from_dict(outlier_summary, orient="index", columns=["outlier_%"]).sort_values("outlier_%", ascending=False)
print("\n--- Potential Outlier Percentages ---")
display(outlier_df.head(15))


In [ ]:
# ===============================================================
# Categorical Features Exploration & Target Relationships
# ===============================================================

# Identify categorical columns
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
print(f"Categorical columns ({len(cat_cols)}): {cat_cols}\n")

# Frequency plots for each categorical variable
for col in cat_cols:
    print(f"🔹 Frequency distribution for: {col}")
    bar_topk_counts(df[col], title=f"{col} value counts (Top 20)")

# Try to identify potential target variable from earlier guess
target_candidates = ["annual_cost", "charges", "expenses", "medical_cost", "expenditure"]
target_col = None
for c in target_candidates:
    if c in df.columns:
        target_col = c
        print(f"✅ Target variable found: {target_col}")
        break

if target_col:
    # Relationship between categorical variables and target (mean cost per category)
    print(f"\n--- Average {target_col} by categorical features ---")
    for col in cat_cols:
        if df[col].nunique() <= 30:  # Only for manageable categories
            grouped = df.groupby(col)[target_col].mean().sort_values(ascending=False)
            display(grouped.head(10))
else:
    print("⚠️ No target variable identified yet. You can specify it manually later.")


In [ ]:
# ===============================================================
# Correlation Analysis (Numeric Relationships)
# ===============================================================

import seaborn as sns

# Identify numeric columns again (in case of previous filtering)
num_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()

# Compute correlation matrix
corr_matrix = df[num_cols].corr(method="spearman")  # Spearman handles non-linear monotonic relations
print("✅ Correlation matrix computed.")

# Display top correlations with target (if identified)
if "target_col" in locals() and target_col:
    target_corr = corr_matrix[target_col].drop(target_col).sort_values(ascending=False)
    print(f"\n🔹 Top correlations with {target_col}:")
    display(target_corr.head(10))
    print(f"\n🔹 Lowest correlations with {target_col}:")
    display(target_corr.tail(10))

# Plot correlation heatmap (top 15 numeric vars if too many)
top_corr_cols = (
    corr_matrix.abs().mean().sort_values(ascending=False).head(15).index
    if len(num_cols) > 15 else num_cols
)

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix.loc[top_corr_cols, top_corr_cols],
            annot=True, fmt=".2f", cmap="coolwarm", square=True)
plt.title("Correlation Heatmap (Top Features)")
plt.show()


In [ ]:
# ===============================================================
# Bivariate Analysis: Predictor vs Target Relationships
# ===============================================================

# Ensure we have a target variable selected
if "target_col" not in locals() or target_col is None:
    # Try to find again if missed earlier
    target_candidates = ["annual_cost", "charges", "expenses", "medical_cost", "expenditure"]
    for c in target_candidates:
        if c in df.columns:
            target_col = c
            print(f"✅ Target variable found automatically: {target_col}")
            break

if target_col:
    # --- Numeric vs Target (scatterplots) ---
    print(f"\n📈 Scatterplots: numeric features vs {target_col}\n")
    numeric_features = [col for col in df.select_dtypes(include=["int64", "float64"]).columns if col != target_col]

    for col in numeric_features:
        fig = px.scatter(
            df,
            x=col,
            y=target_col,
            opacity=0.5,
            trendline="ols",
            title=f"{col} vs {target_col}",
        )
        fig.show()

    # --- Categorical vs Target (boxplots) ---
    print(f"\n📦 Boxplots: categorical features vs {target_col}\n")
    cat_features = df.select_dtypes(include=["object", "category"]).columns.tolist()

    for col in cat_features:
        if df[col].nunique() <= 10:  # only plot manageable categories
            fig = px.box(
                df,
                x=col,
                y=target_col,
                points="all",
                title=f"{col} vs {target_col}",
            )
            fig.show()
else:
    print("⚠️ Target variable not found. Please set target_col manually before running this cell.")


In [ ]:
# ===============================================================
# Final EDA Summary & Data Quality Checks
# ===============================================================

# ---- Duplicate checks ----
n_dupes = df.duplicated().sum()
print(f"🔍 Duplicate rows found: {n_dupes}")

# ---- Zero / negative checks for key numeric columns ----
print("\n⚙️ Checking zero/negative values (potential invalids):")
for col in df.select_dtypes(include=["int64", "float64"]).columns:
    negatives = (df[col] < 0).sum()
    zeros = (df[col] == 0).sum()
    if negatives > 0 or zeros > 0:
        print(f"{col:25s} | zeros: {zeros:6d} | negatives: {negatives:6d}")

# ---- Cardinality summary ----
cardinality = {c: df[c].nunique() for c in df.columns}
card_df = pd.DataFrame(cardinality.items(), columns=["column", "n_unique"]).sort_values("n_unique", ascending=False)
display(card_df.head(15))

# ---- Feature type summary ----
print("\n📋 Feature Type Summary:")
print(f"Numeric features: {len(df.select_dtypes(include=['int64','float64']).columns)}")
print(f"Categorical features: {len(df.select_dtypes(include=['object','category']).columns)}")

# ---- Target distribution summary ----
if "target_col" in locals() and target_col:
    print(f"\n🎯 Target variable summary: {target_col}")
    display(df[target_col].describe(percentiles=[.01, .05, .5, .95, .99]).to_frame().T)
    dist_plot(df[target_col], title=f"{target_col} (final distribution)", log=True)
else:
    print("⚠️ Target variable not identified yet.")

# ---- Save quick EDA summary to file ----
eda_summary = {
    "n_records": int(df.shape[0]),
    "n_features": int(df.shape[1]),
    "n_numeric": int(df.select_dtypes(include=['int64','float64']).shape[1]),
    "n_categorical": int(df.select_dtypes(include=['object','category']).shape[1]),
    "duplicates": int(n_dupes),
    "target_col": target_col if 'target_col' in locals() else None,
    "missing_cols": df.columns[df.isna().any()].tolist(),
    "high_cardinality_cols": card_df[card_df['n_unique'] > 50]['column'].tolist(),
}

with open(REPORT_DIR / "eda_summary.json", "w") as f:
    json.dump(eda_summary, f, indent=2)

print("\n✅ EDA summary report saved to: reports/eda_summary.json")
print(json.dumps(eda_summary, indent=2))


In [ ]:
# ===============================================================
# PHASE 2 - DATA PREPARATION & FEATURE ENGINEERING
# ===============================================================

# Manually set the correct target column
target_col = "annual_medical_cost"   # confirmed target variable

print(f"🎯 Target variable set to: {target_col}")

# Identify feature columns (exclude target)
feature_cols = [c for c in df.columns if c != target_col]

# Separate numeric and categorical features
num_features = df[feature_cols].select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_features = df[feature_cols].select_dtypes(include=["object", "category"]).columns.tolist()

print(f"✅ Feature identification complete:")
print(f"   • Total features: {len(feature_cols)}")
print(f"   • Numeric features: {len(num_features)}")
print(f"   • Categorical features: {len(cat_features)}")
print(f"\nNumeric columns: {num_features[:8]} ...")
print(f"Categorical columns: {cat_features[:8]} ...")


In [ ]:
# ===============================================================
# Handle Missing Values & Data Type Adjustments
# ===============================================================

print("🔍 Checking missing values before cleaning...")
missing = df[feature_cols].isna().sum()
display(missing[missing > 0])

# ---- Step 1: Handle missing numeric values ----
# Replace missing numeric values with median
for col in num_features:
    if df[col].isna().sum() > 0:
        median_value = df[col].median()
        df[col].fillna(median_value, inplace=True)
        print(f"Filled missing numeric values in '{col}' with median ({median_value:.2f})")

# ---- Step 2: Handle missing categorical values ----
# Replace missing categorical values with "Unknown"
for col in cat_features:
    if df[col].isna().sum() > 0:
        df[col].fillna("Unknown", inplace=True)
        print(f"Filled missing categorical values in '{col}' with 'Unknown'")

# ---- Step 3: Convert binary flags to category (if 0/1 columns) ----
binary_candidates = [col for col in num_features if set(df[col].unique()) <= {0, 1}]
if binary_candidates:
    for col in binary_candidates:
        df[col] = df[col].astype("category")
        cat_features.append(col)
        num_features.remove(col)
    print(f"Converted binary columns to categorical: {binary_candidates}")

# ---- Step 4: Verify cleaning ----
print("\n✅ Missing value check after cleaning:")
display(df[feature_cols].isna().sum().sum())
print("✅ Data cleaning complete.")


In [ ]:
# ===============================================================
# Feature Engineering: Encoding & Scaling Pipelines
# ===============================================================

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# --- Numeric pipeline ---
numeric_pipeline = Pipeline(steps=[
    ("scaler", StandardScaler())
])

# --- Categorical pipeline ---
categorical_pipeline = Pipeline(steps=[
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# --- Combined preprocessor ---
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, num_features),
        ("cat", categorical_pipeline, cat_features)
    ],
    remainder="drop"  # drop unused columns
)

print("✅ Preprocessing pipelines created successfully.")
print(f"Numeric features: {len(num_features)}")
print(f"Categorical features: {len(cat_features)}")


In [ ]:
# ===============================================================
# Train / Validation / Test Split Creation
# ===============================================================

from sklearn.model_selection import train_test_split

# Define feature matrix (X) and target vector (y)
X = df[feature_cols]
y = df[target_col]

# For regression, create cost bins to help stratify roughly
y_bins = pd.qcut(y.rank(method="first"), q=10, labels=False, duplicates="drop")

# Split into train + test first
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y_bins
)

# Now split train further into validation
y_train_bins = pd.qcut(y_train.rank(method="first"), q=10, labels=False, duplicates="drop")

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=SEED, stratify=y_train_bins
)

# Display results
print("✅ Data split completed successfully.")
print(f"Train set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set: {X_test.shape}")

# Save indices (for reproducibility)
np.save(PROC_DIR / "train_idx.npy", X_train.index)
np.save(PROC_DIR / "val_idx.npy", X_val.index)
np.save(PROC_DIR / "test_idx.npy", X_test.index)


In [ ]:
# ===============================================================
# Apply Preprocessing Pipeline to Train/Val/Test Sets
# ===============================================================

# Fit the preprocessor only on training data
print("🏗️ Fitting preprocessing pipeline on training data...")
X_train_prep = preprocessor.fit_transform(X_train)

# Apply the same transformations to validation and test data
X_val_prep = preprocessor.transform(X_val)
X_test_prep = preprocessor.transform(X_test)

print("✅ Transformation complete.")
print(f"Transformed training set shape: {X_train_prep.shape}")
print(f"Transformed validation set shape: {X_val_prep.shape}")
print(f"Transformed test set shape: {X_test_prep.shape}")

# Get feature names after transformation (for interpretability)
encoded_feature_names = preprocessor.get_feature_names_out()
X_train_prep = pd.DataFrame(X_train_prep, columns=encoded_feature_names, index=X_train.index)
X_val_prep = pd.DataFrame(X_val_prep, columns=encoded_feature_names, index=X_val.index)
X_test_prep = pd.DataFrame(X_test_prep, columns=encoded_feature_names, index=X_test.index)

# Quick check: ensure no missing or infinite values remain
assert not np.isnan(X_train_prep.values).any(), "NaNs found in training data!"
assert not np.isinf(X_train_prep.values).any(), "Infs found in training data!"

print("✅ Processed datasets are clean and ready for modeling.")


In [ ]:
# ===============================================================
# Domain-Driven Feature Engineering (Fixed)
# ===============================================================

df_fe = df.copy()

# --- 1. Health-related composite features ---
df_fe["bp_ratio"] = df_fe["systolic_bp"] / df_fe["diastolic_bp"]
df_fe["bmi_risk"] = pd.cut(
    df_fe["bmi"],
    bins=[0, 18.5, 25, 30, 35, 100],
    labels=["Underweight", "Normal", "Overweight", "Obese", "Extremely Obese"]
)

# --- 2. Cost efficiency metrics ---
df_fe["claim_efficiency"] = np.where(
    df_fe["claims_count"] > 0,
    df_fe["total_claims_paid"] / df_fe["claims_count"],
    0
)

df_fe["premium_to_cost_ratio"] = (
    df_fe["annual_premium"] / (df_fe["annual_medical_cost"] + 1)
)

# --- 3. Utilization intensity ---
df_fe["proc_total"] = (
    df_fe["proc_imaging_count"] +
    df_fe["proc_surgery_count"] +
    df_fe["proc_physio_count"] +
    df_fe["proc_consult_count"] +
    df_fe["proc_lab_count"]
)

df_fe["visit_intensity"] = (
    df_fe["visits_last_year"] + df_fe["hospitalizations_last_3yrs"]
)

# --- 4. Chronic condition grouping (fix: convert categorical to numeric) ---
chronic_conditions = [
    "hypertension", "diabetes", "asthma", "copd", "cardiovascular_disease",
    "cancer_history", "kidney_disease", "liver_disease", "arthritis", "mental_health"
]

# Convert any categorical or boolean columns to numeric (0/1)
for col in chronic_conditions:
    df_fe[col] = pd.to_numeric(df_fe[col], errors="coerce").fillna(0).astype(int)

# Now safely sum across them
df_fe["chronic_total"] = df_fe[chronic_conditions].sum(axis=1)
df_fe["has_multiple_chronic"] = (df_fe["chronic_total"] >= 2).astype(int)

# --- 5. Interaction features ---
df_fe["age_x_bmi"] = df_fe["age"] * df_fe["bmi"]
df_fe["income_x_risk"] = df_fe["income"] * df_fe["risk_score"]

# --- 6. Update numeric & categorical lists ---
new_numeric = df_fe.select_dtypes(include=["int64", "float64"]).columns.tolist()
new_categorical = df_fe.select_dtypes(include=["object", "category"]).columns.tolist()

print("✅ Feature engineering successful!")
print(f"Numeric columns: {len(new_numeric)} | Categorical columns: {len(new_categorical)}")

# Display some new engineered columns
df_fe[[
    "bp_ratio", "bmi_risk", "claim_efficiency", "premium_to_cost_ratio",
    "proc_total", "chronic_total", "has_multiple_chronic", "age_x_bmi", "income_x_risk"
]].head()


In [ ]:
# ===============================================================
# Update Preprocessing Pipeline with Engineered Features
# ===============================================================

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Identify updated numeric and categorical columns
num_features_fe = df_fe.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_features_fe = df_fe.select_dtypes(include=["object", "category"]).columns.tolist()

# Exclude the target from feature lists
num_features_fe = [c for c in num_features_fe if c != target_col]
cat_features_fe = [c for c in cat_features_fe if c != target_col]

print(f"Updated numeric features: {len(num_features_fe)}")
print(f"Updated categorical features: {len(cat_features_fe)}")

# --- Updated pipelines ---
numeric_pipeline_fe = Pipeline(steps=[
    ("scaler", StandardScaler())
])

categorical_pipeline_fe = Pipeline(steps=[
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# --- Combine numeric + categorical ---
preprocessor_fe = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline_fe, num_features_fe),
        ("cat", categorical_pipeline_fe, cat_features_fe)
    ],
    remainder="drop"
)

# --- Train/val/test split again using engineered dataset ---
X = df_fe.drop(columns=[target_col])
y = df_fe[target_col]

y_bins = pd.qcut(y.rank(method="first"), q=10, labels=False, duplicates="drop")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y_bins
)

y_train_bins = pd.qcut(y_train.rank(method="first"), q=10, labels=False, duplicates="drop")
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=SEED, stratify=y_train_bins
)

# --- Apply preprocessing ---
print("🏗️ Applying preprocessing pipeline to engineered data...")
X_train_prep = preprocessor_fe.fit_transform(X_train)
X_val_prep = preprocessor_fe.transform(X_val)
X_test_prep = preprocessor_fe.transform(X_test)

# Convert back to DataFrames
encoded_feature_names_fe = preprocessor_fe.get_feature_names_out()
X_train_prep = pd.DataFrame(X_train_prep, columns=encoded_feature_names_fe, index=X_train.index)
X_val_prep = pd.DataFrame(X_val_prep, columns=encoded_feature_names_fe, index=X_val.index)
X_test_prep = pd.DataFrame(X_test_prep, columns=encoded_feature_names_fe, index=X_test.index)

print("✅ Engineered data preprocessing complete.")
print(f"Train: {X_train_prep.shape}, Val: {X_val_prep.shape}, Test: {X_test_prep.shape}")


In [ ]:
# ===============================================================
# Save Processed Datasets & Metadata
# ===============================================================

# Save the processed train/validation/test datasets
X_train_prep.to_parquet(PROC_DIR / "X_train_processed.parquet")
X_val_prep.to_parquet(PROC_DIR / "X_val_processed.parquet")
X_test_prep.to_parquet(PROC_DIR / "X_test_processed.parquet")

y_train.to_csv(PROC_DIR / "y_train.csv", index=False)
y_val.to_csv(PROC_DIR / "y_val.csv", index=False)
y_test.to_csv(PROC_DIR / "y_test.csv", index=False)

# Save metadata about the preprocessing
metadata = {
    "target_col": target_col,
    "numeric_features": num_features_fe,
    "categorical_features": cat_features_fe,
    "engineered_features": [
        "bp_ratio", "bmi_risk", "claim_efficiency", "premium_to_cost_ratio",
        "proc_total", "visit_intensity", "chronic_total", "has_multiple_chronic",
        "age_x_bmi", "income_x_risk"
    ],
    "train_shape": X_train_prep.shape,
    "val_shape": X_val_prep.shape,
    "test_shape": X_test_prep.shape
}

with open(PROC_DIR / "feature_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("✅ Processed datasets and metadata saved successfully.")
print(json.dumps(metadata, indent=2))


In [ ]:
# ===============================================================
# PHASE 3 - MODEL DEVELOPMENT & EVALUATION
# ===============================================================

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# --- Load preprocessed data ---
X_train_prep = pd.read_parquet(PROC_DIR / "X_train_processed.parquet")
X_val_prep = pd.read_parquet(PROC_DIR / "X_val_processed.parquet")
X_test_prep = pd.read_parquet(PROC_DIR / "X_test_processed.parquet")

y_train = pd.read_csv(PROC_DIR / "y_train.csv").squeeze()
y_val = pd.read_csv(PROC_DIR / "y_val.csv").squeeze()
y_test = pd.read_csv(PROC_DIR / "y_test.csv").squeeze()

print("✅ Processed data loaded successfully.")
print(f"Train: {X_train_prep.shape}, Validation: {X_val_prep.shape}, Test: {X_test_prep.shape}")

# --- Baseline: Simple Linear Regression ---
linreg = LinearRegression()
linreg.fit(X_train_prep, y_train)

# Validation predictions
y_val_pred = linreg.predict(X_val_prep)

# Evaluation metrics
mae = mean_absolute_error(y_val, y_val_pred)
rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
r2 = r2_score(y_val, y_val_pred)

print("\n📊 Baseline Linear Regression Results (Validation Set):")
print(f"MAE  : {mae:,.2f}")
print(f"RMSE : {rmse:,.2f}")
print(f"R²   : {r2:,.3f}")


In [ ]:
# ===============================================================
# Regularized Linear Models: Ridge, Lasso, ElasticNet
# ===============================================================

from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.model_selection import GridSearchCV

def evaluate_model(model, X, y, name="Model"):
    """Helper function to print evaluation metrics."""
    preds = model.predict(X)
    mae = mean_absolute_error(y, preds)
    rmse = np.sqrt(mean_squared_error(y, preds))
    r2 = r2_score(y, preds)
    print(f"\n📈 {name} Performance:")
    print(f"MAE  : {mae:,.2f}")
    print(f"RMSE : {rmse:,.2f}")
    print(f"R²   : {r2:,.3f}")
    return {"MAE": mae, "RMSE": rmse, "R2": r2}

# --- Ridge Regression ---
ridge_params = {"alpha": [0.1, 1, 10, 50, 100]}
ridge = GridSearchCV(Ridge(random_state=SEED), ridge_params, cv=5, scoring="neg_root_mean_squared_error")
ridge.fit(X_train_prep, y_train)
ridge_best = ridge.best_estimator_

print(f"✅ Best Ridge alpha: {ridge.best_params_}")
ridge_results = evaluate_model(ridge_best, X_val_prep, y_val, "Ridge Regression")

# --- Lasso Regression ---
lasso_params = {"alpha": [0.0001, 0.001, 0.01, 0.1, 1]}
lasso = GridSearchCV(Lasso(random_state=SEED, max_iter=5000), lasso_params, cv=5, scoring="neg_root_mean_squared_error")
lasso.fit(X_train_prep, y_train)
lasso_best = lasso.best_estimator_

print(f"✅ Best Lasso alpha: {lasso.best_params_}")
lasso_results = evaluate_model(lasso_best, X_val_prep, y_val, "Lasso Regression")

# --- ElasticNet Regression ---
enet_params = {
    "alpha": [0.001, 0.01, 0.1, 1],
    "l1_ratio": [0.2, 0.5, 0.8]
}
enet = GridSearchCV(ElasticNet(random_state=SEED, max_iter=5000), enet_params, cv=5, scoring="neg_root_mean_squared_error")
enet.fit(X_train_prep, y_train)
enet_best = enet.best_estimator_

print(f"✅ Best ElasticNet params: {enet.best_params_}")
enet_results = evaluate_model(enet_best, X_val_prep, y_val, "ElasticNet Regression")

# --- Summary comparison ---
results_df = pd.DataFrame([
    {"Model": "Linear Regression", "MAE": mae, "RMSE": rmse, "R2": r2},
    {"Model": "Ridge", **ridge_results},
    {"Model": "Lasso", **lasso_results},
    {"Model": "ElasticNet", **enet_results},
])

display(results_df.sort_values("RMSE"))


In [ ]:
# ===============================================================
# Tree-Based Ensemble Models: RandomForest, XGBoost, LightGBM
# ===============================================================

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# --- Random Forest Regressor ---
rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=3,
    random_state=SEED,
    n_jobs=-1
)
rf_model.fit(X_train_prep, y_train)
rf_results = evaluate_model(rf_model, X_val_prep, y_val, "Random Forest")

# --- XGBoost Regressor ---
xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=SEED,
    n_jobs=-1
)
xgb_model.fit(X_train_prep, y_train, eval_set=[(X_val_prep, y_val)], verbose=False)
xgb_results = evaluate_model(xgb_model, X_val_prep, y_val, "XGBoost")

# --- LightGBM Regressor ---
lgb_model = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=SEED,
    n_jobs=-1
)
lgb_model.fit(X_train_prep, y_train)
lgb_results = evaluate_model(lgb_model, X_val_prep, y_val, "LightGBM")

# --- Summary of all models so far ---
tree_results_df = pd.DataFrame([
    {"Model": "Linear Regression", "MAE": mae, "RMSE": rmse, "R2": r2},
    {"Model": "Ridge", **ridge_results},
    {"Model": "Lasso", **lasso_results},
    {"Model": "ElasticNet", **enet_results},
    {"Model": "Random Forest", **rf_results},
    {"Model": "XGBoost", **xgb_results},
    {"Model": "LightGBM", **lgb_results},
])

print("\n📊 Model Comparison Summary (Validation Set):")
display(tree_results_df.sort_values("RMSE"))


In [ ]:
# ===============================================================
# PHASE 3B - Hyperparameter Tuning & Feature Importance
# ===============================================================
!pip install optuna

import optuna
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor

# Verify XGBoost works
print("✅ XGBoost version check:")
import xgboost
print(xgboost.__version__)

# Optional: reduce Optuna logging verbosity
optuna.logging.set_verbosity(optuna.logging.INFO)

# Sanity check
print(f"Training set shape: {X_train_prep.shape}")
print(f"Validation set shape: {X_val_prep.shape}")


In [ ]:
# ===============================================================
# XGBoost Hyperparameter Optimization with Optuna
# ===============================================================

import optuna
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

# Objective function that Optuna will minimize (RMSE)
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 800),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0.0, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.5, 3.0),
        "random_state": 42,
        "n_jobs": -1,
    }

    model = XGBRegressor(**params)
    model.fit(X_train_prep, y_train, eval_set=[(X_val_prep, y_val)], verbose=False)

    preds = model.predict(X_val_prep)
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    return rmse

# --- Create and run Optuna study ---
print("🔍 Starting XGBoost tuning with Optuna...")
optuna.logging.set_verbosity(optuna.logging.INFO)

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=25, show_progress_bar=True)  # You can increase n_trials for better tuning

print("\n✅ Best trial found:")
print(f"  RMSE: {study.best_value:.3f}")
print(f"  Params: {study.best_params}")


In [ ]:
# ===============================================================
# Train Best XGBoost Model + Feature Importance
# ===============================================================

from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Retrieve best parameters from the Optuna study
best_params = study.best_params
print("✅ Best Parameters for Final Model:")
print(best_params)

# Train final tuned model
best_xgb = XGBRegressor(**best_params)
best_xgb.fit(X_train_prep, y_train)

# Evaluate on validation set
y_pred_val = best_xgb.predict(X_val_prep)
mae = mean_absolute_error(y_val, y_pred_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred_val))
r2 = r2_score(y_val, y_pred_val)

print("\n📊 Tuned XGBoost Validation Performance:")
print(f"MAE  : {mae:,.2f}")
print(f"RMSE : {rmse:,.2f}")
print(f"R²   : {r2:,.3f}")

# --- Feature Importance ---
importances = pd.Series(best_xgb.feature_importances_, index=X_train_prep.columns)
top_feats = importances.sort_values(ascending=False).head(20)

plt.figure(figsize=(8,6))
top_feats.plot(kind="barh")
plt.gca().invert_yaxis()
plt.title("Top 20 Feature Importances (Tuned XGBoost)")
plt.xlabel("Importance Score")
plt.tight_layout()
plt.show()


In [ ]:
# ===============================================================
# Final Evaluation on Test Set + Save Trained Model
# ===============================================================

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import numpy as np
import json

# --- Evaluate tuned model on test data ---
y_pred_test = best_xgb.predict(X_test_prep)

test_mae  = mean_absolute_error(y_test, y_pred_test)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
test_r2   = r2_score(y_test, y_pred_test)

print("\n🏁 Final XGBoost Model Performance (Test Set):")
print(f"MAE  : {test_mae:,.2f}")
print(f"RMSE : {test_rmse:,.2f}")
print(f"R²   : {test_r2:,.3f}")

# --- Save model artifact ---
model_path = MODEL_DIR / "xgboost_tuned_model.joblib"
joblib.dump(best_xgb, model_path)
print(f"\n✅ Model saved to: {model_path}")

# --- Save final evaluation report ---
final_report = {
    "model": "XGBoost (Tuned)",
    "test_metrics": {"MAE": test_mae, "RMSE": test_rmse, "R2": test_r2},
    "best_params": best_params,
}

with open(REPORT_DIR / "final_xgboost_report.json", "w") as f:
    json.dump(final_report, f, indent=2)

print("✅ Final evaluation report saved to: reports/final_xgboost_report.json")


In [ ]:
# ===============================================================
# PHASE 4 - Model Explainability & Insights (SHAP)
# ===============================================================

import shap
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Initialize SHAP explainer for the tuned XGBoost model
print("⚙️ Initializing SHAP explainer...")

explainer = shap.TreeExplainer(best_xgb)
shap_values = explainer.shap_values(X_val_prep)

print("✅ SHAP values computed successfully.")
print(f"Shape of SHAP values: {shap_values.shape}")


In [ ]:
# ===============================================================
# PHASE 4 - Model Explainability & Insights (SHAP KernelExplainer)
# ===============================================================
import shap
import numpy as np
import matplotlib.pyplot as plt

print("⚙️ Initializing SHAP KernelExplainer (model-agnostic)...")

# Take small samples for speed – KernelExplainer is slower
X_background = X_train_prep.sample(100, random_state=42)
X_sample = X_val_prep.sample(300, random_state=42)

# Build the universal SHAP explainer
explainer = shap.KernelExplainer(best_xgb.predict, X_background)

print("⚙️ Calculating SHAP values (this may take a few minutes)...")
shap_values = explainer.shap_values(X_sample, nsamples=100)

# --- Global summary plot ---
shap.summary_plot(shap_values, X_sample, show=False)
plt.tight_layout()
plt.show()

print("✅ SHAP summary plot generated successfully!")


In [ ]:
# ===============================================================
# PHASE 4 - Model Explainability (Stable KernelExplainer Version)
# ===============================================================
import shap
import numpy as np
import matplotlib.pyplot as plt

print("⚙️ Initializing SHAP KernelExplainer (safe lambda version)...")

# Take small samples to make computation manageable
X_background = X_train_prep.sample(100, random_state=42)
X_sample = X_val_prep.sample(300, random_state=42)

# FIX: pass a lambda that calls predict on the model
explainer = shap.KernelExplainer(lambda data: best_xgb.predict(data), X_background)

print("⚙️ Calculating SHAP values (this will take a few minutes)...")
shap_values = explainer.shap_values(X_sample, nsamples=100)

# --- Global summary plot ---
print("📊 Generating SHAP summary plot...")
shap.summary_plot(shap_values, X_sample, show=False)
plt.tight_layout()
plt.show()

print("✅ SHAP summary plot generated successfully!")


In [ ]:
# ===============================================================
# PHASE 4 - SHAP Dependence Analysis (Top Features)
# ===============================================================
import matplotlib.pyplot as plt

print("📈 Generating SHAP dependence plots for top features...")

# Identify the top features from the global importance ranking
top_features = (
    np.abs(shap_values).mean(axis=0)
)
top_feature_indices = np.argsort(top_features)[-5:][::-1]
top_feature_names = X_sample.columns[top_feature_indices]

print(f"Top 5 influential features: {list(top_feature_names)}")

# Generate dependence plots for top features
for feat in top_feature_names:
    shap.dependence_plot(feat, shap_values, X_sample, show=False)
    plt.title(f"SHAP Dependence Plot: {feat}")
    plt.tight_layout()
    plt.show()


In [ ]:
# ===============================================================
# PHASE 4 - Interpretability Report (Narrative Summary)
# ===============================================================

import pandas as pd
import numpy as np

print("🧠 Generating interpretability summary...")

# --- Compute average absolute SHAP values for all features ---
mean_abs_shap = np.abs(shap_values).mean(axis=0)
importance_df = pd.DataFrame({
    "Feature": X_sample.columns,
    "Mean |SHAP|": mean_abs_shap
}).sort_values("Mean |SHAP|", ascending=False).reset_index(drop=True)

# Display top 10 most influential features
print("\n🏆 Top 10 most influential features (by mean |SHAP|):")
display(importance_df.head(10))

# --- Narrative interpretation template ---
summary_text = f"""
🧩 **Phase 4 Interpretability Summary – Key Insights**

1️⃣ **Global Importance**
   The SHAP analysis shows that the following variables contribute most strongly to predicted annual medical costs:
   - {importance_df.iloc[0,0]}
   - {importance_df.iloc[1,0]}
   - {importance_df.iloc[2,0]}
   - {importance_df.iloc[3,0]}
   - {importance_df.iloc[4,0]}

   These features have the highest average absolute SHAP values, meaning they consistently drive the model's output up or down.

2️⃣ **Dependence Observations**
   - Higher values of {importance_df.iloc[0,0]} and {importance_df.iloc[1,0]} tend to **increase** predicted medical costs.
   - Lower values of {importance_df.iloc[2,0]} or {importance_df.iloc[3,0]} are associated with **reduced** expenses.
   - {importance_df.iloc[4,0]} appears to interact with other risk indicators such as chronic_count or risk_score.

3️⃣ **Model Behavior**
   - The model captures **non-linear effects**, where risk or cost escalate sharply beyond certain thresholds.
   - Strong interactions suggest that socioeconomic and health-related factors jointly influence cost prediction.

4️⃣ **Actionable Takeaways**
   - Individuals with high {importance_df.iloc[0,0]} or {importance_df.iloc[1,0]} may benefit from targeted risk interventions.
   - Variables like {importance_df.iloc[2,0]} and {importance_df.iloc[3,0]} could help segment populations by financial exposure.
   - Insurers could use these signals to refine premium calculations or prevention programs.

✅ Overall, SHAP explainability confirms that the model's decisions align well with real-world health and cost drivers.
"""

print(summary_text)


In [ ]:
# ===============================================================
# PHASE 5 - Model Deployment: Save Trained Model & Artifacts
# ===============================================================
import joblib
import json
from datetime import datetime

# --- Create timestamp for versioning ---
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# --- File paths ---
model_path = MODEL_DIR / f"xgboost_tuned_{timestamp}.joblib"
preprocessor_path = MODEL_DIR / f"preprocessor_{timestamp}.joblib"
shap_summary_path = REPORT_DIR / f"shap_summary_{timestamp}.csv"
report_json_path = REPORT_DIR / f"final_model_report_{timestamp}.json"

# --- Save model and preprocessor ---
joblib.dump(best_xgb, model_path)
joblib.dump(preprocessor, preprocessor_path)

print(f"✅ Model saved to: {model_path}")
print(f"✅ Preprocessor saved to: {preprocessor_path}")

# --- Save SHAP summary table for reference ---
importance_df.to_csv(shap_summary_path, index=False)
print(f"✅ SHAP feature importance summary saved to: {shap_summary_path}")

# --- Save performance summary as JSON ---
final_report = {
    "model_type": "XGBoost (Tuned)",
    "timestamp": timestamp,
    "metrics": {
        "train_size": len(X_train_prep),
        "val_size": len(X_val_prep),
        "test_size": len(X_test_prep),
        "mae": round(test_mae, 2),
        "rmse": round(test_rmse, 2),
        "r2": round(test_r2, 3),
    },
    "top_features": list(importance_df["Feature"].head(10))
}

with open(report_json_path, "w") as f:
    json.dump(final_report, f, indent=4)

print(f"✅ Final model report saved to: {report_json_path}")


In [ ]:
# ===============================================================
# PHASE 5 - Deployment Utility: Load Model & Predict
# ===============================================================
import pandas as pd
import joblib

def load_model_and_predict(input_data, model_path, preprocessor_path):
    """
    Load the saved XGBoost model and preprocessing pipeline,
    apply transformations, and return predictions.

    Parameters:
    -----------
    input_data : pd.DataFrame
        New input data for prediction (same column names as training set)
    model_path : str or Path
        Path to the saved .joblib model
    preprocessor_path : str or Path
        Path to the saved preprocessor .joblib file

    Returns:
    --------
    pd.DataFrame : with predicted annual medical costs
    """
    print("⚙️ Loading model and preprocessor...")
    model = joblib.load(model_path)
    preprocessor = joblib.load(preprocessor_path)

    print("🔍 Transforming input data...")
    processed_data = preprocessor.transform(input_data)

    print("🧮 Generating predictions...")
    preds = model.predict(processed_data)

    output = input_data.copy()
    output["Predicted_Annual_Medical_Cost"] = preds

    print("✅ Predictions complete.")
    return output

# Example usage:
# new_data = X_test.sample(5, random_state=42)
# predictions = load_model_and_predict(new_data, model_path, preprocessor_path)
# display(predictions)
